# Football Transfer Market Analyzer - Web Scraping Report

**Author:** Nijat Kazimli  
**Website:** [Transfermarkt](https://www.transfermarkt.com)  
**Competition:** Premier League 2025/26

## What this notebook does

The goal is to build a dataset of Premier League players by scraping Transfermarkt.
For each player I want the basic profile info (name, age, height, position, club,
contract, market value) plus the historical market-value chart shown on the player page.

Tools used:

- **Scrapy** to crawl the league page and each club's squad page
- **requests + BeautifulSoup** for the individual player profile pages
- **Selenium** for the market-value chart, which is rendered by JavaScript
- **regex** for cleaning the messy strings (prices, heights, dates)
- **pandas** to merge and explore the result


## 1. Imports


In [ ]:
import os
import pandas as pd

from regex_utils import (
    parse_market_value, extract_year, extract_date,
    extract_age, extract_height_cm, clean_whitespace,
    extract_player_id, extract_club_id,
)
from profile_scraper import scrape_multiple_profiles
from selenium_scraper import scrape_multiple_market_values
from scrapy_spider import run_scrapy_spider

print("imports ok, pandas", pd.__version__)


All imports successful ✓
Pandas version: 3.0.2
Scrapy version: 2.15.2


## 2. Settings

A few constants. The two MAX_* values are kept small so the notebook runs in a
reasonable time; bump them up for a full crawl.


In [ ]:
BASE_URL = "https://www.transfermarkt.com"
SEASON = "2025"  # 2025/26

MAX_PROFILES_BS4 = 30      # how many profiles to fetch with BeautifulSoup
MAX_PLAYERS_SELENIUM = 10  # how many players to fetch market-value history for

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

print("target:", f"{BASE_URL}/premier-league/ season {SEASON}")
print("profiles to scrape:", MAX_PROFILES_BS4)
print("selenium players:", MAX_PLAYERS_SELENIUM)


Target: https://www.transfermarkt.com/premier-league/ (Season 2025)
Crawl delay: 3s
Max profiles (BS4): 30
Max players (Selenium): 10


## 3. Scrapy: crawl the league table and each club's squad

The spider:

1. starts at the Premier League page
2. picks club links from the league table (`td.hauptlink a[href*='/verein/']`)
3. keeps only the `/startseite/verein/` ones (skips fixture pages)
4. opens each club's full squad page (`/kader/.../plus/1`)
5. reads each row of the squad table

The actual spider class lives in `scrapy_spider.py`. It is run as a subprocess
because Twisted's reactor cannot be restarted in the same Python process
(running it twice in the same Jupyter kernel would otherwise crash).


In [ ]:
# Run the spider (takes ~80s because of the 3s polite delay)
print("running the scrapy spider...\n")

output_path = os.path.join(DATA_DIR, "scrapy_players.json")
scrapy_results = run_scrapy_spider(output_path=output_path, season=SEASON)

print(f"\ngot {len(scrapy_results)} player records -> {output_path}")


Running Scrapy spider to crawl Premier League squads...
(This may take ~80s due to the 3-second crawl delay)

  Launching Scrapy spider subprocess...
  [scrapy] 2026-05-02 20:06:16 [scrapy.utils.log] INFO: Scrapy 2.15.2 started (bot: scrapybot)
  [scrapy] 2026-05-02 20:06:16 [scrapy.utils.log] INFO: Versions:
  [scrapy] {'lxml': '6.1.0',
  [scrapy]  'libxml2': '2.14.6',
  [scrapy]  'cssselect': '1.4.0',
  [scrapy]  'parsel': '1.11.0',
  [scrapy]  'w3lib': '2.4.1',
  [scrapy]  'Twisted': '25.5.0',
  [scrapy]  'Python': '3.14.2 (v3.14.2:df793163d58, Dec  5 2025, 12:18:06) [Clang 16.0.0 '
  [scrapy]            '(clang-1600.0.26.6)]',
  [scrapy]  'pyOpenSSL': '26.1.0 (OpenSSL 4.0.0 14 Apr 2026)',
  [scrapy]  'cryptography': '47.0.0',
  [scrapy]  'Platform': 'macOS-26.4.1-arm64-arm-64bit-Mach-O'}
  [scrapy] 2026-05-02 20:06:16 [scrapy.addons] INFO: Enabled addons:
  [scrapy] []
  [scrapy] 2026-05-02 20:06:16 [scrapy.extensions.telnet] INFO: Telnet Password: 1329c8d6be9d9382
  [scrapy] 2026-

In [ ]:
df_scrapy = pd.DataFrame(scrapy_results)
print("shape:", df_scrapy.shape)
print("columns:", list(df_scrapy.columns))
df_scrapy[["player_name", "club", "position", "market_value_eur"]].head(10)


Scrapy DataFrame shape: (528, 10)

Columns: ['player_name', 'player_id', 'profile_url', 'club', 'position', 'date_of_birth_age', 'nationalities', 'shirt_number', 'market_value_raw', 'market_value_eur']

Sample data:


,player_name,club,position,market_value_eur
0,Max Weiß,Burnley FC,Goalkeeper,4000000.0
1,Martin Dúbravka,Burnley FC,Goalkeeper,750000.0
2,Václav Hladký,Burnley FC,Goalkeeper,200000.0
3,Maxime Estève,Burnley FC,Centre-Back,25000000.0
4,Bashir Humphreys,Burnley FC,Centre-Back,15000000.0
5,Hjalmar Ekdal,Burnley FC,Centre-Back,6000000.0
6,Axel Tuanzebe,Burnley FC,Centre-Back,5000000.0
7,Joe Worrall,Burnley FC,Centre-Back,3000000.0
8,Jordan Beyer,Burnley FC,Centre-Back,2500000.0
9,Quilindschy Hartman,Burnley FC,Left-Back,18000000.0


## 4. requests + BeautifulSoup: player profile pages

For each player URL from Scrapy I `requests.get` the profile page and parse it
with BeautifulSoup to pull out things the squad table doesn't have: full name,
date of birth, place of birth, citizenship, height, preferred foot, contract
dates, international caps/goals.

I sample a few players from each club instead of just taking the first N so
the sample isn't biased towards Arsenal / Aston Villa (which are crawled first
alphabetically).


In [ ]:
# pick a few players per club so the sample covers every team
n_clubs = df_scrapy["club"].nunique()
per_club = max(1, MAX_PROFILES_BS4 // n_clubs)

sampled = (
    df_scrapy.dropna(subset=["profile_url"])
    .groupby("club", group_keys=False)
    .apply(lambda g: g.sample(min(len(g), per_club)))
)
profile_urls = sampled["profile_url"].unique().tolist()[:MAX_PROFILES_BS4]

print(f"scraping {len(profile_urls)} profiles ({per_club} per club, {n_clubs} clubs)\n")

profiles = scrape_multiple_profiles(profile_urls, delay=2.0)
df_profiles = pd.DataFrame(profiles)
print(f"\ndone, {len(df_profiles)} profiles")


Scraping 20 player profiles with requests + BeautifulSoup...
(Sampled evenly across 20 clubs, 1 per club)
Estimated time: ~40s (2s delay per request)

  [1/20] evanilson ✓
  [2/20] max-dowman ✓
  [3/20] ian-maatsen ✓
  [4/20] vitaly-janelt ✓
  [5/20] jason-steele ✓
  [6/20] kyle-walker ✓
  [7/20] marc-guiu ✓
  [8/20] jean-philippe-mateta ✓
  [9/20] harrison-armstrong ✓
  [10/20] emile-smith-rowe ✓
  [11/20] charlie-crew ✓
  [12/20] hugo-ekitike ✓
  [13/20] abdukodir-khusanov ✓
  [14/20] manuel-ugarte ✓
  [15/20] harvey-barnes ✓
  [16/20] ibrahim-sangare ✓
  [17/20] abdoullah-ba ✓
  [18/20] antonin-kinsky ✓
  [19/20] tomas-soucek ✓
  [20/20] joao-gomes ✓

✓ Scraped 20 player profiles with BeautifulSoup


In [ ]:
print("shape:", df_profiles.shape)
print("columns:", list(df_profiles.columns))
df_profiles[["full_name", "age", "height_cm", "position", "foot", "market_value_eur"]].head(10)


Profile DataFrame shape: (20, 17)

Columns: ['profile_url', 'full_name', 'date_of_birth', 'age', 'place_of_birth', 'height_cm', 'citizenship', 'position', 'foot', 'current_club', 'joined', 'contract_expires', 'contract_expires_year', 'market_value_raw', 'market_value_eur', 'international_caps', 'international_goals']


,full_name,age,height_cm,position,foot,market_value_eur
0,Evanilson,26,183,Centre-Forward,both,35000000
1,Max Dowman,16,183,Right Winger,left,20000000
2,Ian Maatsen,24,178,Left-Back,left,30000000
3,Vitaly Janelt,27,184,Defensive Midfield,left,18000000
4,Jason Steele,35,188,Goalkeeper,right,750000
5,Kyle Walker,35,183,Right-Back,right,2500000
6,Marc Guiu,20,187,Centre-Forward,right,12000000
7,Jean-Philippe Mateta,28,192,Centre-Forward,right,35000000
8,Harrison Armstrong,19,185,Central Midfield,right,12000000
9,Emile Smith Rowe,25,182,Attacking Midfield,right,22000000


## 5. Regex helpers

A lot of the fields on Transfermarkt are messy strings (`"€45.00m"`, `"1,95 m"`,
`"21/07/2000 (25)"`). The helpers in `regex_utils.py` clean them up. The cell
below shows what each one does on small examples and then applies the market-value
regex to the real scraped data.


In [ ]:
# quick demo of each helper on hand-picked strings

print("--- parse_market_value ---")
for v in ["€200.00m", "€45.00m", "€500k", "€1.20bn", "-", "€8.00m"]:
    parsed = parse_market_value(v)
    print(f"  {v!r:12} -> {parsed if parsed is None else f'{parsed:,}'}")

print("\n--- extract_year ---")
for t in ["Contract expires: Jun 30, 2026", "Joined: 01/07/2022", "Last extension: 17/01/2025"]:
    print(f"  {t!r} -> {extract_year(t)}")

print("\n--- extract_date ---")
for t in ["Contract expires: Jun 30, 2026", "Joined: 01/07/2022", "Last extension: 17/01/2025"]:
    print(f"  {t!r} -> {extract_date(t)}")

print("\n--- extract_age ---")
for t in ["21/07/2000 (25)", "15/06/1998 (27)", "01/01/1990 (36)"]:
    print(f"  {t!r} -> {extract_age(t)}")

print("\n--- extract_height_cm ---")
for t in ["1,95 m", "1,80 m", "1.73 m", "1,68 m"]:
    print(f"  {t!r} -> {extract_height_cm(t)} cm")

print("\n--- ID extraction from URLs ---")
print(" ", extract_player_id("/erling-haaland/profil/spieler/418560"))
print(" ", extract_club_id("/manchester-city/startseite/verein/281"))

# apply the market-value regex to the real scraped data
print("\n--- applied to scraped data ---")
print("pattern:", r"[€$£]?\s*([\d,.]+)\s*(m|k|bn)?")
if not df_scrapy.empty:
    for raw in df_scrapy["market_value_raw"].dropna().head(5):
        parsed = parse_market_value(raw)
        print(f"  {raw!r:12} -> {'None' if parsed is None else f'€{parsed:,}'}")


REGEX DEMONSTRATIONS

1. parse_market_value():
   '€200.00m' →     200,000,000
   '€45.00m' →      45,000,000
   '€500k' →         500,000
   '€1.20bn' →   1,200,000,000
   '-' → None
   '€8.00m' →       8,000,000

2. extract_year():
   'Contract expires: Jun 30, 2026' → 2026
   'Joined: 01/07/2022' → 2022
   'Last extension: 17/01/2025' → 2025

3. extract_date():
   'Contract expires: Jun 30, 2026' → None
   'Joined: 01/07/2022' → 01/07/2022
   'Last extension: 17/01/2025' → 17/01/2025

4. extract_age():
   '21/07/2000 (25)' → 25
   '15/06/1998 (27)' → 27
   '01/01/1990 (36)' → 36

5. extract_height_cm():
   '1,95 m' → 195 cm
   '1,80 m' → 180 cm
   '1.73 m' → 173 cm
   '1,68 m' → 168 cm

6. extract_player_id() and extract_club_id():
   Player ID: 418560
   Club ID:   281

APPLYING REGEX TO SCRAPED DATA

Market value regex pattern: [€$£]?\s*([\d,.]+)\s*(m|k|bn)?
This matches: currency symbol (optional), number, unit suffix

Raw values from scrape → Parsed:
   '€4.00m' → €4,000,000
   

## 6. Selenium: market value history

The market-value chart on a player page is drawn by a small Svelte widget that
fetches its data from an internal JSON API. That data isn't in the page source,
so plain `requests` won't help.

What I do instead:

1. open the `/marktwertverlauf/` page in headless Chrome (this sets the cookies)
2. accept the cookie banner once
3. from inside the page, run an XHR to the internal API and read the JSON

Doing the XHR from within the page means it's a same-origin request and the
right cookies/headers get sent automatically.


In [ ]:
# take the most expensive players from the scrapy data and fetch their MV history
top_players = (
    df_scrapy.dropna(subset=["market_value_eur"])
    .sort_values("market_value_eur", ascending=False)
    .head(MAX_PLAYERS_SELENIUM)
)

mv_urls = top_players["profile_url"].dropna().unique().tolist()
print(f"fetching market value history for {len(mv_urls)} players...\n")

mv_results = scrape_multiple_market_values(mv_urls, max_players=MAX_PLAYERS_SELENIUM)

# flatten {url: [points]} into a long table
name_by_url = dict(zip(top_players["profile_url"], top_players["player_name"]))
records = []
for url, points in mv_results.items():
    for pt in points:
        pt["player_name"] = name_by_url.get(url, "")
        pt["profile_url"] = url
        records.append(pt)

df_mv_history = pd.DataFrame(records)
print(f"\ngot {len(df_mv_history)} data points")


Scraping market value history for 10 top players via Selenium...

  [1/10] Erling Haaland... ✓ (26 data points)
  [2/10] Declan Rice... ✓ (24 data points)
  [3/10] Bukayo Saka... ✓ (23 data points)
  [4/10] Cole Palmer... ✓ (16 data points)
  [5/10] Florian Wirtz... ✓ (17 data points)
  [6/10] Moisés Caicedo... ✓ (19 data points)
  [7/10] Dominik Szoboszlai... ✓ (26 data points)
  [8/10] Alexander Isak... ✓ (29 data points)
  [9/10] Hugo Ekitiké... ✓ (18 data points)
  [10/10] Enzo Fernández... ✓ (17 data points)

✓ Selenium collected 215 market value data points


In [ ]:
if df_mv_history.empty:
    print("no market value data collected")
else:
    print("shape:", df_mv_history.shape)
    display(df_mv_history[["player_name", "date_display", "value_raw", "value_eur", "club_at_time"]].head(15))


Market Value History DataFrame shape: (215, 8)

Sample data points:


,player_name,date_display,value_raw,value_eur,club_at_time
0,Erling Haaland,2016-12-18,€200.00K,200000,Bryne FK
1,Erling Haaland,2017-12-23,€300.00K,300000,Molde FK
2,Erling Haaland,2018-09-10,€2.00M,2000000,Molde FK
3,Erling Haaland,2018-12-30,€5.00M,5000000,Molde FK
4,Erling Haaland,2019-06-03,€5.00M,5000000,Red Bull Salzburg
5,Erling Haaland,2019-09-03,€12.00M,12000000,Red Bull Salzburg
6,Erling Haaland,2019-11-07,€30.00M,30000000,Red Bull Salzburg
7,Erling Haaland,2019-12-16,€45.00M,45000000,Red Bull Salzburg
8,Erling Haaland,2020-02-11,€60.00M,60000000,Borussia Dortmund
9,Erling Haaland,2020-03-11,€80.00M,80000000,Borussia Dortmund


## 7. Put everything together

There are three data sources now:

- the Scrapy squad data (one row per player, basic squad fields)
- the BeautifulSoup profile data (deeper attributes for a sample of players)
- the Selenium market-value history (time series for the top players)

The first two get merged on `profile_url` into a single player-level table.
The history stays as a separate long table because it's many rows per player.


In [ ]:
if not df_profiles.empty and "profile_url" in df_profiles.columns:
    df_players = df_scrapy.merge(
        df_profiles, on="profile_url", how="left", suffixes=("_squad", "_profile")
    )
else:
    df_players = df_scrapy.copy()

print("merged shape:", df_players.shape)
print(f"\n{len(df_players.columns)} columns:")
for col in sorted(df_players.columns):
    print(f"  {col:35s} {df_players[col].notna().sum():>4} non-null")

df_players.head()


Merged players DataFrame: (528, 26)

All columns (26):
  age                                   20 non-null values
  citizenship                           20 non-null values
  club                                 528 non-null values
  contract_expires                      20 non-null values
  contract_expires_year                 19 non-null values
  current_club                          20 non-null values
  date_of_birth                         20 non-null values
  date_of_birth_age                    528 non-null values
  foot                                  20 non-null values
  full_name                             20 non-null values
  height_cm                             20 non-null values
  international_caps                    20 non-null values
  international_goals                   20 non-null values
  joined                                20 non-null values
  market_value_eur_profile              20 non-null values
  market_value_eur_squad               524 non-null values
 

,player_name,player_id,profile_url,club,position_squad,date_of_birth_age,nationalities,shirt_number,market_value_raw_squad,market_value_eur_squad,...,position_profile,foot,current_club,joined,contract_expires,contract_expires_year,market_value_raw_profile,market_value_eur_profile,international_caps,international_goals
0,Max Weiß,829299,https://www.transfermarkt.com/max-weiss/profil...,Burnley FC,Goalkeeper,15/06/2004 (21),[Germany],13,€4.00m,4000000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Martin Dúbravka,74960,https://www.transfermarkt.com/martin-dubravka/...,Burnley FC,Goalkeeper,15/01/1989 (37),[Slovakia],1,€750k,750000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Václav Hladký,95795,https://www.transfermarkt.com/vaclav-hladky/pr...,Burnley FC,Goalkeeper,14/11/1990 (35),[Czech Republic],32,€200k,200000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Maxime Estève,842341,https://www.transfermarkt.com/maxime-esteve/pr...,Burnley FC,Centre-Back,26/05/2002 (23),[France],5,€25.00m,25000000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Bashir Humphreys,612517,https://www.transfermarkt.com/bashir-humphreys...,Burnley FC,Centre-Back,15/03/2003 (23),"[England, Uganda]",12,€15.00m,15000000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 8. Clean up and look at the data

A few quick fixes before analysis:

- bucket the detailed positions ("Left Winger", "Centre-Back", ...) into four
  big groups (GK / DEF / MID / FWD)
- coerce the market value to a numeric column
- add an age group column
- drop duplicate `player_id`s (sometimes the same player appears twice if a
  squad page lists him in two sections)


In [ ]:
# map the detailed positions onto 4 broad categories
position_map = {
    "Goalkeeper": "Goalkeeper",
    "Centre-Back": "Defender",
    "Left-Back": "Defender",
    "Right-Back": "Defender",
    "Defensive Midfield": "Midfielder",
    "Central Midfield": "Midfielder",
    "Attacking Midfield": "Midfielder",
    "Left Midfield": "Midfielder",
    "Right Midfield": "Midfielder",
    "Left Winger": "Forward",
    "Right Winger": "Forward",
    "Centre-Forward": "Forward",
    "Second Striker": "Forward",
}

pos_col = "position" if "position" in df_players.columns else "position_squad"
df_players["position_category"] = df_players[pos_col].map(position_map).fillna("Unknown")

# market value -> numeric
mv_col = "market_value_eur_squad" if "market_value_eur_squad" in df_players.columns else "market_value_eur"
df_players["market_value_eur_clean"] = pd.to_numeric(df_players[mv_col], errors="coerce")

# age column (the bs4 profile has it; the squad table doesn't)
if "age" in df_players.columns:
    df_players["age_clean"] = pd.to_numeric(df_players["age"], errors="coerce")
else:
    df_players["age_clean"] = pd.NA

# age groups
df_players["age_group"] = pd.cut(
    df_players["age_clean"],
    bins=[0, 21, 25, 29, 33, 45],
    labels=["U21", "22-25", "26-29", "30-33", "34+"],
)

df_players = df_players.drop_duplicates(subset=["player_id"], keep="first")

# summary
print("total unique players:", len(df_players))
print("\nposition breakdown:")
print(df_players["position_category"].value_counts().to_string())
print("\nage groups:")
print(df_players["age_group"].value_counts().sort_index().to_string())
print("\nmarket value stats (EUR):")
print(df_players["market_value_eur_clean"].describe().apply(lambda x: f"€{x:,.0f}"))
print("\nclubs represented:", df_players["club"].nunique())

if not df_mv_history.empty:
    print("\nmarket value history:")
    print("  data points:", len(df_mv_history))
    print("  players:    ", df_mv_history["player_name"].nunique())
    if "date_display" in df_mv_history.columns:
        print(f"  date range:  {df_mv_history['date_display'].min()} .. {df_mv_history['date_display'].max()}")


DATASET SUMMARY

Total unique players: 528

Position distribution:
position_category
Defender      183
Midfielder    143
Forward       137
Goalkeeper     65

Age group distribution:
age_group
U21      4
22-25    8
26-29    5
30-33    1
34+      2

Market value statistics (EUR):
count            €524
mean      €24,000,620
std       €22,700,739
min           €75,000
25%        €8,000,000
50%       €20,000,000
75%       €32,000,000
max      €200,000,000
Name: market_value_eur_clean, dtype: str

Clubs represented: 20

--- Market Value History ---
Total data points: 215
Players covered: 10
Date range: 2016-06-17 → 2026-03-09


In [ ]:
# top 20 most valuable players
print("top 20 most valuable PL players\n")
top20 = (
    df_players.nlargest(20, "market_value_eur_clean")
    [["player_name", "club", "position_category", "age_clean", "market_value_eur_clean"]]
    .reset_index(drop=True)
)
top20.index += 1
top20.columns = ["Player", "Club", "Position", "Age", "Market Value (EUR)"]
top20["Market Value (EUR)"] = top20["Market Value (EUR)"].apply(
    lambda x: f"€{x/1_000_000:.1f}M" if pd.notna(x) else "N/A"
)
print(top20.to_string())

# average market value by position
print("\n\naverage market value by position:")
by_pos = (
    df_players.groupby("position_category")["market_value_eur_clean"]
    .agg(["mean", "median", "count"])
    .sort_values("mean", ascending=False)
)
by_pos["mean"] = by_pos["mean"].apply(lambda x: f"€{x/1_000_000:.1f}M")
by_pos["median"] = by_pos["median"].apply(lambda x: f"€{x/1_000_000:.1f}M")
print(by_pos.to_string())

# average market value by age group
print("\n\naverage market value by age group:")
by_age = (
    df_players.groupby("age_group", observed=True)["market_value_eur_clean"]
    .agg(["mean", "median", "count"])
    .sort_index()
)
by_age["mean"] = by_age["mean"].apply(lambda x: f"€{x/1_000_000:.1f}M")
by_age["median"] = by_age["median"].apply(lambda x: f"€{x/1_000_000:.1f}M")
print(by_age.to_string())


TOP 20 MOST VALUABLE PLAYERS IN THE PREMIER LEAGUE
                 Player               Club    Position   Age Market Value (EUR)
1        Erling Haaland    Manchester City     Forward   NaN            €200.0M
2           Declan Rice         Arsenal FC  Midfielder   NaN            €120.0M
3           Bukayo Saka         Arsenal FC     Forward   NaN            €120.0M
4         Florian Wirtz       Liverpool FC  Midfielder   NaN            €110.0M
5        Moisés Caicedo         Chelsea FC  Midfielder   NaN            €110.0M
6           Cole Palmer         Chelsea FC  Midfielder   NaN            €110.0M
7    Dominik Szoboszlai       Liverpool FC  Midfielder   NaN            €100.0M
8        Alexander Isak       Liverpool FC     Forward   NaN            €100.0M
9      Ryan Gravenberch       Liverpool FC  Midfielder   NaN             €90.0M
10         Hugo Ekitiké       Liverpool FC     Forward  23.0             €90.0M
11       Enzo Fernández         Chelsea FC  Midfielder   NaN         

## 9. Save the results


In [ ]:
# player-level table
players_csv = os.path.join(DATA_DIR, "players_full.csv")
df_players.to_csv(players_csv, index=False, encoding="utf-8-sig")
print(f"saved {players_csv}  ({len(df_players)} rows, {len(df_players.columns)} cols)")

# market value history (time series)
if not df_mv_history.empty:
    mv_csv = os.path.join(DATA_DIR, "market_value_history.csv")
    df_mv_history.to_csv(mv_csv, index=False, encoding="utf-8-sig")
    print(f"saved {mv_csv}  ({len(df_mv_history)} rows)")

# raw scrapy json was already written by the spider
print(f"raw scrapy json: {os.path.join(DATA_DIR, 'scrapy_players.json')}")

print("\nfiles in data/:")
for f in sorted(os.listdir(DATA_DIR)):
    p = os.path.join(DATA_DIR, f)
    if os.path.isfile(p):
        print(f"  {f:40s} {os.path.getsize(p):>10,} bytes")

print(f"\nfinal player dataframe: {df_players.shape}")


✓ Saved: data/players_full.csv (528 rows, 30 columns)
✓ Saved: data/market_value_history.csv (215 rows)
✓ Scrapy raw JSON: data/scrapy_players.json

FINAL OUTPUT FILES
  market_value_history.csv                     29,113 bytes
  players_full.csv                            115,020 bytes
  scrapy_players.json                         189,623 bytes

✓ Project complete. Final DataFrame shape: (528, 30)
  Ready for analysis: correlating age, position, and market valuation.


## Wrap-up

What ended up being used where:

- **Scrapy** crawls the league page, follows every club, parses each squad table.
  Respects `robots.txt` and uses a 3-second delay.
- **requests + BeautifulSoup** picks up the extra profile fields that aren't in
  the squad table (height, foot, contract, citizenship, caps).
- **Selenium** is only needed for the market-value chart, because that one is
  rendered by JavaScript from an internal JSON endpoint.
- **regex** does the small parsing jobs: `"€45.00m" -> 45000000`, `"1,95 m" -> 195`,
  pulling years out of contract strings, etc.
- **pandas** merges the three sources, fills in derived columns (age group,
  position category) and writes the CSVs.

Output files:

- `data/players_full.csv` - one row per player, all attributes merged
- `data/market_value_history.csv` - time series of valuations for the top players
- `data/scrapy_players.json` - the raw Scrapy output
